**Building ensemble classifiers models from MotherCV**
---------------------------------------------
This notebook shows how to
1. Build an ensemble model using the estimator output from `mother_cv` using `CVEnsembleClassifierMother`
2. Predict labels and uncertainty with the estimator using `predict_uncertainty()`

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, train_test_split
import numpy as np

from mother import ml
from mother.ml.ensembles.cvEnsemble import CVEnsembleClassifierMother
import mother.pipeline_utils as mother_takes_care

# Prepare data

In [2]:
df_data, df_targets = load_breast_cancer(return_X_y=True, as_frame=True)
print(f"Downloaded dataset with {df_data.shape[0]} samples and {df_data.shape[1]} features")
print(f"The target we want to predict has {len(np.unique(df_targets))} unique classes: {np.unique(df_targets)}")

Downloaded dataset with 569 samples and 30 features
The target we want to predict has 2 unique classes: [0 1]


In [3]:
X_train, X_test, y_train, y_test = train_test_split(df_data, df_targets, test_size=0.2, random_state=42)

print(f"Training set has {X_train.shape[0]} samples and {X_train.shape[1]} features")
print(f"Test set has {X_test.shape[0]} samples and {X_test.shape[1]} features")

Training set has 455 samples and 30 features
Test set has 114 samples and 30 features


# Cross-Validation

In [4]:
model = ml.CatboostClassifierMother(random_seed=42)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

df_cv, estimators_dict = mother_takes_care.mother_cv(
    estimator=model, X=X_train, y=y_train, cv=cv, return_estimators=True
)

0:	learn: 0.6542782	total: 55.5ms	remaining: 55.4s
1:	learn: 0.6199305	total: 65.3ms	remaining: 32.6s
2:	learn: 0.5779254	total: 81ms	remaining: 26.9s
3:	learn: 0.5438688	total: 92.8ms	remaining: 23.1s
4:	learn: 0.5138768	total: 100ms	remaining: 19.9s
5:	learn: 0.4770701	total: 109ms	remaining: 18s
6:	learn: 0.4504077	total: 123ms	remaining: 17.4s
7:	learn: 0.4239091	total: 130ms	remaining: 16.1s
8:	learn: 0.4009386	total: 138ms	remaining: 15.1s
9:	learn: 0.3775747	total: 144ms	remaining: 14.3s
10:	learn: 0.3556669	total: 151ms	remaining: 13.6s
11:	learn: 0.3378300	total: 161ms	remaining: 13.2s
12:	learn: 0.3202937	total: 174ms	remaining: 13.2s
13:	learn: 0.3049027	total: 207ms	remaining: 14.5s
14:	learn: 0.2935436	total: 217ms	remaining: 14.2s
15:	learn: 0.2793748	total: 226ms	remaining: 13.9s
16:	learn: 0.2644487	total: 238ms	remaining: 13.7s
17:	learn: 0.2521326	total: 246ms	remaining: 13.4s
18:	learn: 0.2430144	total: 255ms	remaining: 13.1s
19:	learn: 0.2314076	total: 263ms	remaini

In [5]:
df_cv.head()

,pred_target,pred_proba_0,pred_proba_1,pred_knowledge_uncertainty,pred_data_uncertainty,pred_total_uncertainty,target,iteration,test_index
68,1,0.396323,0.603677,0.010003,0.607297,0.6173,1,0,0
181,0,0.999922,0.000078,0.000006,0.001224,0.001229,0,4,1
63,1,0.000302,0.999698,0.000019,0.003931,0.00395,1,2,2
248,1,0.002761,0.997239,0.000046,0.023393,0.023439,1,1,3
60,1,0.000916,0.999084,0.000081,0.011182,0.011263,1,3,4


In [6]:
estimators_dict

{'estimators': [CatboostClassifierMother(auto_class_weights='Balanced', boosting_type='Plain', bootstrap_type='MVS', grow_policy='SymmetricTree', learning_rate=0.03, loss_function='Logloss', max_depth=6, posterior_sampling=True, random_seed=42, random_strength=1),
  CatboostClassifierMother(auto_class_weights='Balanced', boosting_type='Plain', bootstrap_type='MVS', grow_policy='SymmetricTree', learning_rate=0.03, loss_function='Logloss', max_depth=6, posterior_sampling=True, random_seed=42, random_strength=1),
  CatboostClassifierMother(auto_class_weights='Balanced', boosting_type='Plain', bootstrap_type='MVS', grow_policy='SymmetricTree', learning_rate=0.03, loss_function='Logloss', max_depth=6, posterior_sampling=True, random_seed=42, random_strength=1),
  CatboostClassifierMother(auto_class_weights='Balanced', boosting_type='Plain', bootstrap_type='MVS', grow_policy='SymmetricTree', learning_rate=0.03, loss_function='Logloss', max_depth=6, posterior_sampling=True, random_seed=42, ra

# Predict with ensemble model

In [14]:
ensemble = CVEnsembleClassifierMother(estimators=estimators_dict["estimators"])

ensemble_prediction = ensemble.predict_uncertainty(X=X_test, uncertainty_method="disagreement")

## Evaluation

In [15]:
y_pred = ensemble_prediction["pred"]

# compute accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9737


In [16]:
ensemble_prediction["knowledge_uncertainty"].describe()

count    114.000000
mean       0.032988
std        0.134158
min        0.000004
25%        0.000045
50%        0.000251
75%        0.002863
max        1.129094
Name: knowledge_uncertainty, dtype: float64